In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

DATASETS = [
    "Bank_Customer_Churn_Dataset","cmc","connect-4","credit-g",
    "elevators","eucalyptus","eye_movements","house_16H","kc1",
    "phoneme","pol","socmob","splice","vehicle",
    "california_housing","cpu_act","Bike_Sharing_Demand",
    "abalone","wine_quality"
]

DATA_PATH = "data"

results = []

for dataset in DATASETS:

    try:

        base = Path(DATA_PATH) / dataset

        N = None
        C = None

        if (base / "N_train.npy").exists():
            N = np.load(
                base / "N_train.npy",
                allow_pickle=True
            )

        if (base / "C_train.npy").exists():
            C = np.load(
                base / "C_train.npy",
                allow_pickle=True
            )

        if N is not None and C is not None:
            X = np.hstack([N, C])

        elif N is not None:
            X = N

        elif C is not None:
            X = C

        else:
            print(f"Skipping {dataset}: no data")
            continue

        df = pd.DataFrame(X)

        total_cells = df.shape[0] * df.shape[1]

        missing_count = int(
            df.isna().sum().sum()
        )

        missing_pct = (
            missing_count / total_cells * 100
            if total_cells > 0 else 0
        )

        missing_per_col = df.isna().sum()

        cols_with_missing = int(
            (missing_per_col > 0).sum()
        )

        max_missing_col = (
            missing_per_col.max()
            if len(missing_per_col) > 0
            else 0
        )

        if pd.isna(max_missing_col):
            max_missing_col = 0

        results.append({
            "dataset": dataset,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "missing_values": missing_count,
            "missing_pct": round(missing_pct, 4),
            "cols_with_missing": cols_with_missing,
            "max_missing_in_one_col": int(max_missing_col)
        })

    except Exception as e:

        print(
            f"ERROR on {dataset}: {e}"
        )

summary = pd.DataFrame(results)

summary = summary.sort_values(
    "missing_pct",
    ascending=False
)

print("\n====================")
print("MISSING VALUES AUDIT")
print("====================\n")

print(summary)

print("\n====================")
print("DATASETS WITH MISSING")
print("====================\n")

print(
    summary[
        summary["missing_values"] > 0
    ][[
        "dataset",
        "missing_values",
        "missing_pct",
        "cols_with_missing"
    ]]
)

summary.to_csv(
    "missing_values_summary.csv",
    index=False
)

print("\nSaved: missing_values_summary.csv")

Skipping california_housing: no data
Skipping cpu_act: no data
Skipping Bike_Sharing_Demand: no data
Skipping abalone: no data
Skipping wine_quality: no data

MISSING VALUES AUDIT

                        dataset   rows  cols  missing_values  missing_pct  \
5                    eucalyptus    470    19             333        3.729   
0   Bank_Customer_Churn_Dataset   6400    10               0        0.000   
1                           cmc    942     9               0        0.000   
2                     connect-4  43236    42               0        0.000   
3                      credit-g    640    20               0        0.000   
4                     elevators  10623    18               0        0.000   
6                 eye_movements   6998    27               0        0.000   
7                     house_16H   8632    16               0        0.000   
8                           kc1   1349    21               0        0.000   
9                       phoneme   3458     5     

In [4]:
from pathlib import Path

for d in DATASETS[:3]:

    base = Path(DATA_PATH)/d

    print("\nDATASET:", d)

    for f in sorted(base.iterdir()):
        print(f.name)


DATASET: Bank_Customer_Churn_Dataset
C_test.npy
C_test.npy:Zone.Identifier
C_test.npy:Zone.Identifier:Zone.Identifier
C_train.npy
C_train.npy:Zone.Identifier
C_train.npy:Zone.Identifier:Zone.Identifier
C_train.npy:Zone.Identifier:Zone.Identifier:Zone.Identifier
C_val.npy
C_val.npy:Zone.Identifier
C_val.npy:Zone.Identifier:Zone.Identifier
C_val.npy:Zone.Identifier:Zone.Identifier:Zone.Identifier
N_test.npy
N_test.npy:Zone.Identifier
N_test.npy:Zone.Identifier:Zone.Identifier
N_test.npy:Zone.Identifier:Zone.Identifier:Zone.Identifier
N_train.npy
N_train.npy:Zone.Identifier
N_train.npy:Zone.Identifier:Zone.Identifier
N_train.npy:Zone.Identifier:Zone.Identifier:Zone.Identifier
N_val.npy
N_val.npy:Zone.Identifier
N_val.npy:Zone.Identifier:Zone.Identifier
N_val.npy:Zone.Identifier:Zone.Identifier:Zone.Identifier
info.json
info.json:Zone.Identifier
info.json:Zone.Identifier:Zone.Identifier
y_test.npy
y_test.npy:Zone.Identifier
y_test.npy:Zone.Identifier:Zone.Identifier
y_train.npy
y_train.np

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import (
    mannwhitneyu,
    chi2_contingency
)

ALPHA = 0.05


def analyze_missingness(df, target_col):

    results = []

    missing_col = f"{target_col}_missing"

    df = df.copy()

    df[missing_col] = (
        df[target_col]
        .isna()
        .astype(int)
    )

    for col in df.columns:

        if col in [target_col, missing_col]:
            continue

        try:

            # NUMERIC FEATURE
            if pd.api.types.is_numeric_dtype(df[col]):

                g0 = df.loc[
                    df[missing_col] == 0,
                    col
                ].dropna()

                g1 = df.loc[
                    df[missing_col] == 1,
                    col
                ].dropna()

                if len(g0) < 5 or len(g1) < 5:
                    continue

                stat, p = mannwhitneyu(
                    g0,
                    g1,
                    alternative="two-sided"
                )

                results.append({
                    "feature": col,
                    "test": "mannwhitney",
                    "pvalue": p,
                    "significant": p < ALPHA
                })

            # CATEGORICAL FEATURE
            else:

                table = pd.crosstab(
                    df[col].astype(str),
                    df[missing_col]
                )

                if table.shape[0] < 2:
                    continue

                chi2, p, _, _ = chi2_contingency(
                    table
                )

                results.append({
                    "feature": col,
                    "test": "chi_square",
                    "pvalue": p,
                    "significant": p < ALPHA
                })

        except Exception:
            pass

    res = pd.DataFrame(results)

    if len(res):

        res = res.sort_values(
            "pvalue"
        )

        n_sig = (
            res["significant"]
            .sum()
        )

        print(
            f"\n{target_col}"
        )

        print(
            f"Significant associations: "
            f"{n_sig}/{len(res)}"
        )

        print(
            res.head(20)
        )

    return res